In [47]:
import celldega as dega
from glob import glob
import pandas as pd
import tifffile

/Users/feni/Documents/celldega/dega/lib/python3.12/site-packages/h5py/__init__.py:36: UserWarning: h5py is running against HDF5 1.14.5 when it was built against 1.14.6, this may cause problems
  _warn(("h5py is running against HDF5 {0} when it was built against {1}, "


In [43]:
dataset_name = 'E12_71'
path_data = 'data/IST_data/Substrate_' + dataset_name + '/'
path_landscape_files = 'data/IST_landscape_files/'

In [38]:
inst_slice = 'T8'
image_scale = 1.0
suffix = '.webp[Q=100]'

## Image

In [45]:
# Path to your OME-TIFF file
file_path = path_data + 'registered_images/' + inst_slice + '_' + dataset_name + '.ome.tiff'

# Open the OME-TIFF file and read the image data
with tifffile.TiffFile(file_path) as tif:
    series = tif.series[0] 
    image_data = series.asarray()

In [49]:
# image_data_scaled = image_data[:,:0] * 2
# Save the image data to a regular TIFF file without compression
tifffile.imwrite(path_landscape_files + 'output_regular.tif', image_data, compression=None)
# image_ds = dega.pre.reduce_image_size(path_landscape_files + 'output_regular.tif', image_scale, path_landscape_files)
image_png = dega.pre._convert_to_png(path_landscape_files + 'output_regular.tif')
dega.pre.make_deepzoom_pyramid(image_png, path_landscape_files + 'pyramid_images/', 'h&e', suffix=suffix)

# Spots

In [16]:
tsv_file = path_data + 'Substrate_E14_62_map_file.tsv' 

'data/IST_data/Substrate_E12_71/Substrate_E14_62_map_file.tsv'

In [29]:
# Define parameters
tsv_file = path_data + 'Substrate_E12_71_map_file.tsv' 
chunk_size = 10_000_000
parquet_prefix = path_landscape_files + 'map_parquet_files/output_chunk'

for i, chunk in enumerate(pd.read_csv(tsv_file, sep="\t", chunksize=chunk_size, header=None, index_col=0)):
    output_file = f"{parquet_prefix}_{i}.parquet"
    chunk.index.name = None
    chunk.to_parquet(output_file, engine="pyarrow")

    if i%20 == 0:
        print(f"Saved {output_file}")

print("Processing complete!")

# Region Barcodes

In [31]:
barcodes = pd.read_csv(
    path_data + 'matrix_files/T1_E12_71/T1_E12_71_raw/barcodes.tsv.gz', 
    sep='\t', 
    header=None, 
    index_col=0
)
barcodes.index.name = None
barcodes['x'] = pd.Series(index=barcodes.index.tolist())
barcodes['y'] = pd.Series(index=barcodes.index.tolist())

In [34]:
barcodes_list = barcodes.index.tolist()

for inst_file in glob(path_landscape_files + 'map_parquet_files/*.parquet'):
    
    inst_chunk = pd.read_parquet(inst_file)

    common_barcodes = list(set(inst_chunk.index.tolist()).intersection(barcodes_list))

    print(inst_file, 'found', len(common_barcodes), 'barcodes')

    if len(common_barcodes) > 0:
        barcodes.loc[common_barcodes, 'x'] = inst_chunk.loc[common_barcodes, 1]
        barcodes.loc[common_barcodes, 'y'] = inst_chunk.loc[common_barcodes, 2]
        

data/IST_landscape_files/map_parquet_files/output_chunk_50.parquet found 0 barcodes
data/IST_landscape_files/map_parquet_files/output_chunk_40.parquet found 0 barcodes
data/IST_landscape_files/map_parquet_files/output_chunk_2.parquet found 0 barcodes
data/IST_landscape_files/map_parquet_files/output_chunk_32.parquet found 0 barcodes
data/IST_landscape_files/map_parquet_files/output_chunk_22.parquet found 2344732 barcodes
data/IST_landscape_files/map_parquet_files/output_chunk_49.parquet found 0 barcodes
data/IST_landscape_files/map_parquet_files/output_chunk_59.parquet found 0 barcodes
data/IST_landscape_files/map_parquet_files/output_chunk_14.parquet found 0 barcodes
data/IST_landscape_files/map_parquet_files/output_chunk_58.parquet found 0 barcodes
data/IST_landscape_files/map_parquet_files/output_chunk_48.parquet found 0 barcodes
data/IST_landscape_files/map_parquet_files/output_chunk_15.parquet found 0 barcodes
data/IST_landscape_files/map_parquet_files/output_chunk_41.parquet foun

In [35]:
barcodes.to_parquet(path_landscape_files + 'meta_spots.parquet')

## Cells

In [ ]:
cells = pd.read_csv(
    '/Volumes/T9/michal_data/Substrate_E14_62/matrix_files/' + inst_slice + '_E14_62/' + inst_slice + '_E14_62_cell_binned/barcodes.tsv.gz', 
    sep='\t', 
    header=None, 
    index_col=0
)